# 12. More outlets in training (added 8/11)

**Question.** The limit seems to be the number of independent outlets (about 180 per fold). If the non-political
NewsGuard outlets we scraped are added to the TRAINING (the test stays the 226 political outlets, same folds and same
draws), do the models predict the political outlets better?

Everything is in this notebook: the extra dataset (same cleaning and anonymization rules as steps 2-3, copies of political
articles removed, ownership groups kept isolated, at most 30 articles per extra outlet), the frozen vectors of the extra
articles, the models (TF-IDF, Qwen3-8B and Mistral-24B with ridge, outlet-weighted ridge and the small neural network),
the comparison with the same models without extra outlets (steps 7 and 10). Rules and reasons in `more_outlets.py`.
The GPU part waits for the step 11 queue to finish (`WAIT_FOR`). Every piece is saved, so a re-run resumes.

In [ ]:
N_JOBS = 48
RUN_BERT = False                      # ModernBERT on meta-articles with the extra outlets (as step 11), only on request
WAIT_FOR = "Results/meta_done"        # the GPU part starts after this file exists (step 11 queue finished)

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt

import anonymization as AN
import boost as BO
import boost_eval as BE
import more_outlets as MX

MX.OUT.mkdir(parents=True, exist_ok=True)
def safely(label, f, *a):
    """A failing piece is logged and the notebook goes on (a re-run retries only what is missing)."""
    try:
        f(*a)
    except Exception as e:
        C.log(f"ERROR {label}: {type(e).__name__}: {e}", MX.LOG)
        with open(MX.OUT / "errors.txt", "a") as fh:
            fh.write(f"==== {label}\n{traceback.format_exc()}\n")

## 12.1 The extra dataset

In [ ]:
steps = []
def step(name, df):
    steps.append({"step": name, "articles": len(df), "outlets": df.domain.nunique()})
    print(f"{name}: {len(df):,} articles, {df.domain.nunique():,} outlets")

final = MX.OUT / "articles_extra_anonymous.csv.gz"
if not final.exists():
    ng, cand = MX.candidates()
    print(f"candidate outlets (NewsGuard score, not political-only): {len(cand):,}")
    raw = MX.extract(cand, N_JOBS); step("scraped articles of the candidates", raw)
    art = MX.filters(raw, N_JOBS, step)
    political = pd.read_csv(C.OUT / "articles.csv.gz")
    art = MX.deduplicate(art, political, N_JOBS, step)
    art = MX.groups_and_cap(art, sorted(political.domain.unique()), ng, step)
    anon, counts, found = AN.anonymize(art)
    left = AN.leftovers(anon, found)
    RULES = ["signature", "email", "url", "account", "author", "name"]
    print(f"anonymization: articles touched {(counts[RULES].sum(axis=1) > 0).mean():.1%}; "
          f"distinctive names left as a word: {left[~left.common_word].left_as_word.sum()}")
    anon[["article_id", "domain", "title", "date", "text", "score", "label"]].to_csv(final, index=False)
    pd.DataFrame(steps).to_csv(MX.OUT / "steps.csv", index=False)
pd.read_csv(MX.OUT / "steps.csv").set_index("step")

In [ ]:
extra = pd.read_csv(final)
U, O, rows, outlets, extra_outlets = MX.union()
eo = O.loc[extra_outlets]
print(f"extra: {len(extra_outlets)} outlets, {len(extra):,} articles; political: {len(outlets)} outlets, {(U.article_id < MX.EXTRA_ID).sum():,} articles")
pd.DataFrame({"political outlets": outlets.label.value_counts(), "extra outlets": eo.label.value_counts()}).reindex(range(5)).rename(index=dict(enumerate(C.LEVELS))).T

## 12.2 TF-IDF (CPU)

In [ ]:
for f in range(1, 6):
    safely(f"TF-IDF fold {f}", MX.fit_tfidf, f, U, rows, outlets, extra_outlets, N_JOBS)

## 12.3 Frozen vectors and models (GPU, after the step 11 queue)

In [ ]:
while not Path(WAIT_FOR).exists():
    time.sleep(300)
for name in ["Qwen3-8B", "Mistral-24B"]:
    holder = {}
    safely(f"embed {name}", lambda: holder.update(X=MX.embed(name, U)))
    if "X" not in holder:
        continue
    X = holder["X"]
    for method in ["ridge", "ridge_outlet", "mlp"]:
        for f in range(1, 6):
            safely(f"{name} · {method} fold {f}", MX.fit_embedding, name, method, f, X, U, O, rows, outlets, extra_outlets)
    del X

In [ ]:
if RUN_BERT:
    import meta as ME
    M, _ = ME.load_meta()
    art_pol, _, _ = BO.load()
    for f in range(1, 6):
        safely(f"ModernBERT fold {f}", ME.fit_bert, f, M, art_pol, outlets, extra, extra.groupby("domain").score.first(), MX.PRED)

## 12.4 With and without the extra outlets

In [ ]:
done = [m for m in MX.MODELS if len(list((MX.PRED / m).glob("fold*.csv.gz"))) == 5]
print("complete:", done)
err = MX.OUT / "errors.txt"
print(open(err).read()[-3000:] if err.exists() else "no errors")
idx = np.vstack([np.arange(len(outlets))[None, :], np.random.default_rng(7).integers(0, len(outlets), (BE.B, len(outlets)))])
rows_out = []
for m in done:
    for k in ["1", "5", "all"]:
        with_ = BE._one(MX.matrix(m, outlets, k, True), m, k, outlets, idx)[2]
        without = BE._one(MX.matrix(m, outlets, k, False), m, k, outlets, idx)[2]
        for (score, mes), v in with_.items():
            d = v - without[(score, mes)]; lo, hi = np.percentile(d[1:], [2.5, 97.5])
            rows_out.append({"model": m, "k": k, "score": score, "measure": mes, "without": without[(score, mes)][0],
                             "with extra": v[0], "difference": d[0], "low": lo, "high": hi,
                             "better": bool(hi < 0) if mes in BE.LOWER else bool(lo > 0),
                             "worse": bool(lo > 0) if mes in BE.LOWER else bool(hi < 0)})
T = pd.DataFrame(rows_out); T.to_csv(MX.OUT / "results.csv", index=False)

In [ ]:
def table(k="all", score="stretched"):
    """without → with extra outlets; ✓ significantly better, ✗ significantly worse (paired bootstrap)."""
    d = T[(T.k == k) & (T.score == score)].copy()
    f = lambda v, m: f"{v:.3f}" if m in ("Spearman", "Kendall") else f"{v:.1f}"
    d["text"] = [f"{f(a, m)} → {f(b, m)}" + (" ✓" if g else " ✗" if w else "") for a, b, m, g, w in
                 zip(d.without, d["with extra"], d.measure, d.better, d.worse)]
    return d.pivot_table(index="model", columns="measure", values="text", aggfunc="first")[BE.MEASURES].reindex(done)


table("all", "stretched")

In [ ]:
table("all", "normal")

In [ ]:
table("5", "stretched")

In [ ]:
table("1", "stretched")

## In short

In [ ]:
d = T[(T.k == "all") & (T.score == "stretched")]
print(d.pivot_table(index="model", columns="measure", values=["without", "with extra"]).round(3).to_string())
print("\nsignificantly better with extra outlets:", d.better.sum(), "| worse:", d.worse.sum(), "| of", len(d))